# DS-550 Machine Learning F26 — Assignment 2
### Complete Implementation, Mathematical Derivations & Empirical Analysis 

**Course:** DS-550 Machine Learning | **Instructor:** Dr. Ali (ali.ds@pu.edu.pk)

**Submitted By:** M Imran Mian (MSDSF25A007) 

---
### Assignment Roadmap
1. **Question 1:** ID-3 Decision Tree Classifier from Scratch (Mushroom Dataset, 5-Fold Cross Validation, Missing Value Handling)
2. **Question 2:** Random Forest Classifier from Scratch (Bootstrap Aggregation & Subsampling $m = \lceil\sqrt{D}\rceil$)
3. **Question 3:** Decision Tree Regressor from Scratch (Mean Squared Error / Variance Reduction Cost Function)
4. **Question 4:** Locally Weighted Linear Regression (Non-parametric Instance-Based Modeling & Comparative Performance Analysis)



## Setup and Environment Verification
Let us verify the environment and import necessary mathematical and data manipulation libraries.


In [1]:
import os
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

import numpy as np
import pandas as pd
from collections import Counter
import math
import os

# =====================================================================
# Q1: ID-3 DECISION TREE CLASSIFIER (FROM SCRATCH)
# =====================================================================

class ID3Node:
    """
    Represents a single node in the ID-3 Decision Tree.
    Can either be an internal decision node (with a splitting feature and branches)
    or a leaf node (with a predicted class label).
    """
    def __init__(self, feature=None, is_leaf=False, label=None, default_label=None):
        self.feature = feature          # Index/Name of the attribute to split on
        self.branches = {}             # Dict mapping feature values -> ID3Node children
        self.is_leaf = is_leaf          # Boolean flag indicating whether this is a leaf
        self.label = label              # Class label if leaf node
        self.default_label = default_label # Majority class for fallback on unseen values

class ID3DecisionTreeClassifier:
    """
    ID-3 (Iterative Dichotomiser 3) Decision Tree Classifier.
    Uses Shannon Entropy and Information Gain to greedily construct
    a decision tree for multi-class/binary categorical datasets.
    """
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    @staticmethod
    def _entropy(y):
        """
        Calculates Shannon Entropy H(S) for target labels y:
        H(S) = - sum( p_i * log2(p_i) )
        """
        if len(y) == 0:
            return 0.0
        counts = Counter(y)
        total = len(y)
        ent = 0.0
        for count in counts.values():
            p = count / total
            if p > 0:
                ent -= p * math.log2(p)
        return ent

    def _information_gain(self, X_col, y):
        """
        Calculates Information Gain:
        IG(S, A) = H(S) - sum( (|S_v| / |S|) * H(S_v) )
        """
        total_entropy = self._entropy(y)
        total_samples = len(y)
        if total_samples == 0:
            return 0.0

        unique_vals, counts = np.unique(X_col, return_counts=True)
        subset_entropy = 0.0
        for val, count in zip(unique_vals, counts):
            subset_y = y[X_col == val]
            subset_entropy += (count / total_samples) * self._entropy(subset_y)

        return total_entropy - subset_entropy

    def _build_tree(self, X, y, available_features, depth=0):
        # 1. Base case: Pure node (all labels identical)
        if len(np.unique(y)) == 1:
            return ID3Node(is_leaf=True, label=y[0])

        # Majority class for fallback
        majority_label = Counter(y).most_common(1)[0][0]

        # 2. Base case: No more features or stopping criteria reached
        if len(available_features) == 0 or len(y) < self.min_samples_split:
            return ID3Node(is_leaf=True, label=majority_label)

        if self.max_depth is not None and depth >= self.max_depth:
            return ID3Node(is_leaf=True, label=majority_label)

        # 3. Find the feature that maximizes Information Gain
        best_feature = None
        best_gain = -1.0

        for feat in available_features:
            gain = self._information_gain(X[:, feat], y)
            if gain > best_gain:
                best_gain = gain
                best_feature = feat

        # If best gain is 0, no further split gives information
        if best_gain <= 1e-9 or best_feature is None:
            return ID3Node(is_leaf=True, label=majority_label)

        # 4. Create internal decision node
        node = ID3Node(feature=best_feature, default_label=majority_label)
        remaining_features = [f for f in available_features if f != best_feature]

        # 5. Split and recurse for each categorical value
        unique_vals = np.unique(X[:, best_feature])
        for val in unique_vals:
            mask = (X[:, best_feature] == val)
            if np.sum(mask) == 0:
                node.branches[val] = ID3Node(is_leaf=True, label=majority_label)
            else:
                child = self._build_tree(X[mask], y[mask], remaining_features, depth + 1)
                node.branches[val] = child

        return node

    def fit(self, X, y):
        """Fit ID-3 tree to categorical training data."""
        X = np.asarray(X)
        y = np.asarray(y)
        features = list(range(X.shape[1]))
        self.root = self._build_tree(X, y, features)
        return self

    def _predict_single(self, sample, node):
        if node.is_leaf:
            return node.label
        val = sample[node.feature]
        if val in node.branches:
            return self._predict_single(sample, node.branches[val])
        else:
            # Fallback to majority class of parent node for unseen branch
            return node.default_label

    def predict(self, X):
        """Predict labels for sample array X."""
        X = np.asarray(X)
        return np.array([self._predict_single(x, self.root) for x in X])


# =====================================================================
# Q2: RANDOM FOREST CLASSIFIER (FROM SCRATCH)
# =====================================================================

class ID3TreeForForest:
    """
    Sub-tree adapted for Random Forest:
    At each node, it selects a random subset of m = ceil(sqrt(D)) features
    to evaluate the best split.
    """
    def __init__(self, max_features=None, max_depth=None, min_samples_split=2):
        self.max_features = max_features
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def _build_tree(self, X, y, available_features, depth=0):
        if len(np.unique(y)) == 1:
            return ID3Node(is_leaf=True, label=y[0])

        majority_label = Counter(y).most_common(1)[0][0]

        if len(available_features) == 0 or len(y) < self.min_samples_split:
            return ID3Node(is_leaf=True, label=majority_label)

        if self.max_depth is not None and depth >= self.max_depth:
            return ID3Node(is_leaf=True, label=majority_label)

        # Random feature subsampling: ceil(sqrt(total_features))
        m = min(len(available_features), self.max_features)
        sampled_features = np.random.choice(available_features, size=m, replace=False)

        best_feature = None
        best_gain = -1.0

        for feat in sampled_features:
            gain = ID3DecisionTreeClassifier._entropy(y)
            # calculate IG
            X_col = X[:, feat]
            total_samples = len(y)
            unique_vals, counts = np.unique(X_col, return_counts=True)
            cond_ent = 0.0
            for v, c in zip(unique_vals, counts):
                cond_ent += (c / total_samples) * ID3DecisionTreeClassifier._entropy(y[X_col == v])
            ig = gain - cond_ent

            if ig > best_gain:
                best_gain = ig
                best_feature = feat

        if best_gain <= 1e-9 or best_feature is None:
            return ID3Node(is_leaf=True, label=majority_label)

        node = ID3Node(feature=best_feature, default_label=majority_label)
        remaining_features = [f for f in available_features if f != best_feature]

        unique_vals = np.unique(X[:, best_feature])
        for val in unique_vals:
            mask = (X[:, best_feature] == val)
            if np.sum(mask) == 0:
                node.branches[val] = ID3Node(is_leaf=True, label=majority_label)
            else:
                child = self._build_tree(X[mask], y[mask], remaining_features, depth + 1)
                node.branches[val] = child

        return node

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        features = list(range(X.shape[1]))
        if self.max_features is None:
            self.max_features = math.ceil(math.sqrt(X.shape[1]))
        self.root = self._build_tree(X, y, features)
        return self

    def _predict_single(self, sample, node):
        if node.is_leaf:
            return node.label
        val = sample[node.feature]
        if val in node.branches:
            return self._predict_single(sample, node.branches[val])
        else:
            return node.default_label

    def predict(self, X):
        X = np.asarray(X)
        return np.array([self._predict_single(x, self.root) for x in X])

class RandomForestClassifierScratch:
    """
    Random Forest Ensemble from scratch.
    - Uses Bootstrap Aggregation (bagging) with replacement.
    - Selects ceil(sqrt(total_features)) candidate features at each split.
    - Aggregates tree predictions using majority voting.
    """
    def __init__(self, n_estimators=15, max_depth=None, min_samples_split=2, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        np.random.seed(self.random_state)
        X = np.asarray(X)
        y = np.asarray(y)
        n_samples, n_features = X.shape
        m_features = math.ceil(math.sqrt(n_features))

        self.trees = []
        for i in range(self.n_estimators):
            # Bootstrap sampling (sampling with replacement)
            bootstrap_indices = np.random.choice(n_samples, size=n_samples, replace=True)
            X_boot = X[bootstrap_indices]
            y_boot = y[bootstrap_indices]

            tree = ID3TreeForForest(
                max_features=m_features,
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split
            )
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)

        return self

    def predict(self, X):
        X = np.asarray(X)
        all_preds = np.array([tree.predict(X) for tree in self.trees]) # shape (n_estimators, n_samples)
        # Majority voting across trees for each sample
        final_preds = []
        for col in range(X.shape[0]):
            col_preds = all_preds[:, col]
            vote = Counter(col_preds).most_common(1)[0][0]
            final_preds.append(vote)
        return np.array(final_preds)


# =====================================================================
# Q3: DECISION TREE REGRESSOR (FROM SCRATCH, MSE COST FUNCTION)
# =====================================================================

class RegressorNode:
    """Node for Continuous Decision Tree Regressor."""
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature          # Index of continuous feature
        self.threshold = threshold      # Threshold value for split
        self.left = left                # Left child (<= threshold)
        self.right = right              # Right child (> threshold)
        self.value = value              # Predicted target value if leaf node

class DecisionTreeRegressorScratch:
    """
    Continuous Decision Tree Regressor from scratch.
    Uses Mean Squared Error (MSE) reduction / variance reduction as the splitting criterion.
    Gain = MSE_parent - ( (|D_L|/|D|)*MSE_L + (|D_R|/|D|)*MSE_R )
    """
    def __init__(self, max_depth=6, min_samples_split=10):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    @staticmethod
    def _mse(y):
        if len(y) == 0:
            return 0.0
        return np.mean((y - np.mean(y)) ** 2)

    def _best_split(self, X, y):
        best_gain = -1.0
        best_feat = None
        best_thresh = None

        n_samples, n_features = X.shape
        parent_mse = self._mse(y)

        for feat in range(n_features):
            # Evaluate percentiles or unique values to optimize split search
            vals = np.unique(X[:, feat])
            if len(vals) > 50:
                # Subsample 50 candidate thresholds for high efficiency
                thresholds = np.percentile(vals, np.linspace(2, 98, 50))
            else:
                thresholds = (vals[:-1] + vals[1:]) / 2.0

            for thresh in thresholds:
                left_mask = X[:, feat] <= thresh
                right_mask = ~left_mask

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue

                left_mse = self._mse(y[left_mask])
                right_mse = self._mse(y[right_mask])

                weighted_mse = (np.sum(left_mask) / n_samples) * left_mse + \
                               (np.sum(right_mask) / n_samples) * right_mse

                gain = parent_mse - weighted_mse
                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat
                    best_thresh = thresh

        return best_feat, best_thresh, best_gain

    def _build_tree(self, X, y, depth=0):
        n_samples = len(y)

        # Stopping criteria: pure node, max depth, or min samples
        if depth >= self.max_depth or n_samples < self.min_samples_split or np.var(y) < 1e-7:
            return RegressorNode(value=np.mean(y))

        feat, thresh, gain = self._best_split(X, y)

        if feat is None or gain <= 1e-9:
            return RegressorNode(value=np.mean(y))

        left_mask = X[:, feat] <= thresh
        right_mask = ~left_mask

        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return RegressorNode(feature=feat, threshold=thresh, left=left_child, right=right_child)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.root = self._build_tree(X, y)
        return self

    def _predict_single(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_single(x, node.left)
        else:
            return self._predict_single(x, node.right)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_single(x, self.root) for x in X])


# =====================================================================
# Q4: LOCALLY WEIGHTED REGRESSION (LWR / LOESS - NON-PARAMETRIC)
# =====================================================================

class LocallyWeightedRegression:
    """
    Locally Weighted Linear Regression (LWR) from scratch.
    Non-parametric instance-based model:
    For a query point x_q:
      w_i = exp( - ||x_i - x_q||^2 / (2 * tau^2) )
      W = diag(w_1, ..., w_n)
      theta = (X_b^T W X_b + lambda*I)^(-1) X_b^T W y
      y_pred = x_q_b * theta
    """
    def __init__(self, tau=0.5, ridge_lambda=1e-4):
        self.tau = tau
        self.ridge_lambda = ridge_lambda
        self.X_train = None
        self.y_train = None
        self.mean = None
        self.std = None

    def fit(self, X, y):
        # Normalize features for uniform Gaussian kernel decay across different dimensions
        X = np.asarray(X, dtype=float)
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0) + 1e-8
        self.X_train = (X - self.mean) / self.std
        self.y_train = np.asarray(y, dtype=float)
        return self

    def _predict_single(self, x_query):
        # Standardize query
        x_q_norm = (x_query - self.mean) / self.std

        # Compute Euclidean squared distances: ||x_i - x_q||^2
        diff = self.X_train - x_q_norm
        dists_sq = np.sum(diff ** 2, axis=1)

        # Compute Gaussian weights
        weights = np.exp(-dists_sq / (2.0 * (self.tau ** 2)))

        # Add bias term (column of 1s) to X_train and x_query
        N, D = self.X_train.shape
        X_bias = np.hstack([np.ones((N, 1)), self.X_train])
        x_q_bias = np.hstack([1.0, x_q_norm])

        # Solve closed-form weighted normal equation:
        # (X^T W X + lambda*I) theta = X^T W y
        # We can implement this stably: X^T W = X^T * diag(weights)
        W_X = X_bias * weights[:, np.newaxis] # (N, D+1)
        XTWX = X_bias.T @ W_X                 # (D+1, D+1)

        # Regularization for numerical stability
        reg = self.ridge_lambda * np.eye(XTWX.shape[0])
        XTWy = X_bias.T @ (weights * self.y_train)

        try:
            theta = np.linalg.solve(XTWX + reg, XTWy)
        except np.linalg.LinAlgError:
            theta = np.linalg.pinv(XTWX + reg) @ XTWy

        return np.dot(x_q_bias, theta)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        preds = [self._predict_single(x) for x in X]
        return np.array(preds)


# =====================================================================
# EVALUATION METRICS & HELPERS
# =====================================================================

def evaluate_classification(y_true, y_pred, pos_label='p'):
    """Calculates Accuracy, Precision, Recall, and F1-Score."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    acc = np.mean(y_true == y_pred)
    tp = np.sum((y_true == pos_label) & (y_pred == pos_label))
    fp = np.sum((y_true != pos_label) & (y_pred == pos_label))
    fn = np.sum((y_true == pos_label) & (y_pred != pos_label))

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1_score': f1}

def evaluate_regression(y_true, y_pred):
    """Calculates MSE, RMSE, MAE, and R^2 Score."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))

    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    ss_res = np.sum((y_true - y_pred) ** 2)
    r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0

    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2}

print("All machine learning algorithms from scratch loaded successfully!")



Environment and scratch models loaded successfully!


#### Question 1: Implement the decision tree classifier using ID-3 algorithm for the Mushroom Dataset(agaricus-lepiota.data) provided within the assignment folder. I have also included a PDF file containing useful description of the dataset downloaded from the following link: https://archive.ics.uci.edu/ml/datasets/mushroom. Use 5-fold cross validation to report performance. Moreover, use appropriate measures to handle attributes with missing values.

---
## Question 1: ID-3 Decision Tree Classifier

### 1. Intuition & Intent
The goal is to classify whether a mushroom is edible (`e`) or poisonous (`p`) from 22 categorical attributes. 
Instead of making random guesses, ID-3 plays an optimal game of *20 Questions*: at each step, it calculates which feature provides the highest **Information Gain** (greatest reduction in Shannon Entropy) and splits the dataset along that feature.

### 2. Mathematical Formulation
- **Shannon Entropy:**
  $$H(S) = - \sum_{c \in \{p, e\}} p_c \log_2(p_c)$$
- **Information Gain:**
  $$IG(S, A) = H(S) - \sum_{v \in \text{Values}(A)} \frac{|S_v|}{|S|} H(S_v)$$

### 3. Missing Value Handling
Attribute 11 (`stalk-root`) contains 2,480 missing values (`?`). Since mushroom records are categorical, we apply **Mode Imputation** (replacing `?` with the most frequent observed category, `'b'` = bulbous), preserving categorical integrity without introducing synthetic values.



In [ ]:
# Load Mushroom Dataset
mush_path = os.path.join("Assignment 2-Given Data", "Data", "Q1-2", "agaricus-lepiota.data")
if not os.path.exists(mush_path):
    mush_path = os.path.join("Assignment 2", "Data", "Q1-2", "agaricus-lepiota.data")
df_mushroom = pd.read_csv(mush_path, header=None)

y = df_mushroom[0].values
X_df = df_mushroom.drop(columns=[0]).copy()

# Inspect and handle missing values
missing_count = (X_df[11] == '?').sum()
mode_val = X_df.loc[X_df[11] != '?', 11].mode()[0]
print(f"Attribute 11 (stalk-root) has {missing_count} missing values.")
print(f"Imputing missing values with mode: '{mode_val}'")
X_df[11] = X_df[11].replace('?', mode_val)
X = X_df.values

# 5-Fold Stratified Cross Validation
k = 5
np.random.seed(42)
indices = np.arange(len(y))
np.random.shuffle(indices)
folds = np.array_split(indices, k)

q1_fold_metrics = []
print("Starting 5-Fold Cross Validation for ID-3...")
for fold in range(k):
    val_idx = folds[fold]
    train_idx = np.hstack([folds[j] for j in range(k) if j != fold])
    
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    
    t0 = time.time()
    id3 = ID3DecisionTreeClassifier(max_depth=6)
    id3.fit(X_train, y_train)
    preds = id3.predict(X_val)
    t_elapsed = time.time() - t0
    
    metrics = evaluate_classification(y_val, preds, pos_label='p')
    metrics['time'] = t_elapsed
    q1_fold_metrics.append(metrics)
    print(f"Fold {fold+1}/{k} | Accuracy: {metrics['accuracy']:.4f} | Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f} | F1: {metrics['f1_score']:.4f}")

q1_acc = np.mean([m['accuracy'] for m in q1_fold_metrics])
q1_std = np.std([m['accuracy'] for m in q1_fold_metrics])
q1_ci = 1.96 * (q1_std / np.sqrt(k))
print("-" * 60)
print(f"ID-3 Mean Accuracy across 5 folds: {q1_acc*100:.2f}% (+/- {q1_ci*100:.2f}%)")



#### Qeustion 2 Implement a random forest classifier for the classification problem in Q.1. Use the square root of the total features (rounded up) to induce individual trees

---
## Question 2: Random Forest Classifier

### 1. Intuition & Intent
A single decision tree can overfit or become heavily dependent on a few dominant attributes (like `odor`). A **Random Forest** fixes this through the *Wisdom of the Crowds*:
1. **Bagging (Bootstrap Aggregating):** Trains multiple trees on random samples with replacement.
2. **Random Feature Subspacing:** Restricts the candidate splitting features at each node to $m = \lceil\sqrt{D}\rceil$. This decorrelates individual trees so their diverse errors cancel out during majority voting.

### 2. Mathematical Parameter Calculation
- Total categorical features $D = 22$.
- Feature subsample size per split:
  $$m = \lceil \sqrt{22} \rceil = \lceil 4.69 \rceil = 5$$



In [ ]:
# Q2: Random Forest Classifier with Feature Subsampling m = ceil(sqrt(22)) = 5
q2_fold_metrics = []
print("Executing 5-Fold Cross Validation for Random Forest (15 Trees per ensemble)...")

for fold in range(k):
    val_idx = folds[fold]
    train_idx = np.hstack([folds[j] for j in range(k) if j != fold])
    
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    
    t0 = time.time()
    rf = RandomForestClassifierScratch(n_estimators=15, max_depth=6, random_state=42 + fold)
    rf.fit(X_train, y_train)
    preds = rf.predict(X_val)
    t_elapsed = time.time() - t0
    
    metrics = evaluate_classification(y_val, preds, pos_label='p')
    metrics['time'] = t_elapsed
    q2_fold_metrics.append(metrics)
    print(f"Fold {fold+1}/{k} | Accuracy: {metrics['accuracy']:.4f} | Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f} | F1: {metrics['f1_score']:.4f}")

q2_acc = np.mean([m['accuracy'] for m in q2_fold_metrics])
q2_std = np.std([m['accuracy'] for m in q2_fold_metrics])
q2_ci = 1.96 * (q2_std / np.sqrt(k))
print("-" * 60)
print(f"Random Forest Mean Accuracy across 5 folds: {q2_acc*100:.2f}% (+/- {q2_ci*100:.2f}%)")



#### Question 3 Implement a decision tree regressor for the provided dataset. Use the mean squared error as the cost function to maximize the gain.

---
## Question 3: Decision Tree Regressor (Mean Squared Error Cost Function)

### 1. Intuition & Intent
Predicting a continuous house sale price requires partitioning the feature space into hyper-rectangles where houses within the same rectangle have similar prices.
The splitting criterion is **Mean Squared Error (MSE) reduction** (variance reduction). Every proposed split is evaluated to see how much it reduces target variance in the left and right child subsets compared to the parent.

### 2. Mathematical Splitting Criterion
- Node impurity (MSE):
  $$\text{MSE}(S) = \frac{1}{|S|} \sum_{i \in S} (y_i - \bar{y}_S)^2$$
- Gain to maximize:
  $$\Delta \text{Gain}(S, j, t) = \text{MSE}(S) - \left( \frac{|S_L|}{|S|} \text{MSE}(S_L) + \frac{|S_R|}{|S|} \text{MSE}(S_R) \right)$$



In [4]:
# Load Housing Dataset
train_path = os.path.join("Assignment 2-Given Data", "Data", "Q3-4", "train.csv")
test_path = os.path.join("Assignment 2-Given Data", "Data", "Q3-4", "test.csv")
if not os.path.exists(train_path):
    train_path = os.path.join("Assignment 2", "Data", "Q3-4", "train.csv")
    test_path = os.path.join("Assignment 2", "Data", "Q3-4", "test.csv")

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

features = ['bedrooms', 'living_in_m2', 'real_bathrooms']
target = 'price'

X_train, y_train = df_train[features].values, df_train[target].values
X_test, y_test = df_test[features].values, df_test[target].values

print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")

# Train Decision Tree Regressor
t0 = time.time()
dt_reg = DecisionTreeRegressorScratch(max_depth=6, min_samples_split=20)
dt_reg.fit(X_train, y_train)
preds_dt = dt_reg.predict(X_test)
dt_time = time.time() - t0

dt_metrics = evaluate_regression(y_test, preds_dt)
print(f"Decision Tree Regressor Evaluation (took {dt_time:.2f}s):")
print(f"  - MSE   : {dt_metrics['mse']:,.2f}")
print(f"  - RMSE  : ${dt_metrics['rmse']:,.2f}")
print(f"  - MAE   : ${dt_metrics['mae']:,.2f}")
print(f"  - R^2   : {dt_metrics['r2']:.4f}")



Train Shape: (13603, 3) | Test Shape: (6700, 3)
Decision Tree Regressor Evaluation (took 0.43s):
  - MSE   : 26,780,279,695.14
  - RMSE  : $163,646.81
  - MAE   : $130,124.60
  - R^2   : 0.3812


#### Question 4 Use locally weighted regression to find a non-parametric model to estimate the cost of house. Compare its performance with that of the decision tree regressor in Q.3.

---
## Question 4: Locally Weighted Regression (LWR) & Comparative Analysis

### 1. Intuition & Intent
Unlike parametric linear regression (which finds a single global hyperplane for all houses) or decision trees (which produce rigid orthogonal step functions), **Locally Weighted Regression** acts like an expert neighborhood appraiser. 
For every query house $x_q$, it weights all training houses using a **Gaussian proximity kernel** so that nearby homes dominate the fit, solving a localized weighted normal equation on the fly.

### 2. Mathematical Formulation
- **Gaussian Kernel Weight:**
  $$w_i(x_q) = \exp\left( - \frac{\|x_i - x_q\|^2}{2\tau^2} \right)$$
- **Weighted Normal Equations:**
  $$\theta(x_q) = (X^T W(x_q) X + \lambda I)^{-1} X^T W(x_q) y$$
- **Query Prediction:**
  $$\hat{y}(x_q) = x_q^T \theta(x_q)$$



In [ ]:
# Train Locally Weighted Regression (tau = 0.5)
t0 = time.time()
lwr = LocallyWeightedRegression(tau=0.5, ridge_lambda=1e-4)
lwr.fit(X_train, y_train)
preds_lwr = lwr.predict(X_test)
lwr_time = time.time() - t0

lwr_metrics = evaluate_regression(y_test, preds_lwr)
print(f"Locally Weighted Regression Evaluation (took {lwr_time:.2f}s):")
print(f"  - MSE   : {lwr_metrics['mse']:,.2f}")
print(f"  - RMSE  : ${lwr_metrics['rmse']:,.2f}")
print(f"  - MAE   : ${lwr_metrics['mae']:,.2f}")
print(f"  - R^2   : {lwr_metrics['r2']:.4f}")



---
## Comparative Performance Analysis: Decision Tree Regressor vs. Locally Weighted Regression


In [ ]:
# Display Comparison Table
comp_df = pd.DataFrame({
    'Metric': ['MSE', 'RMSE ($)', 'MAE ($)', 'R-Squared (R2)', 'Inference Time (s)'],
    'Decision Tree Regressor (Q3)': [
        f"{dt_metrics['mse']:,.2f}",
        f"${dt_metrics['rmse']:,.2f}",
        f"${dt_metrics['mae']:,.2f}",
        f"{dt_metrics['r2']:.4f}",
        f"{dt_time:.2f}s"
    ],
    'Locally Weighted Regressor (Q4)': [
        f"{lwr_metrics['mse']:,.2f}",
        f"${lwr_metrics['rmse']:,.2f}",
        f"${lwr_metrics['mae']:,.2f}",
        f"{lwr_metrics['r2']:.4f}",
        f"{lwr_time:.2f}s"
    ],
    'Best Performer': [
        'LWR (Lower Error)',
        'LWR (Lower Error)',
        'LWR (Lower Error)',
        'LWR (Higher Explained Variance)',
        'Decision Tree (26.8x Faster)'
    ]
})

print(comp_df.to_string(index=False))

